We're gonna import the dataset we inspected in notebook 01.

In [33]:
from datasets import load_dataset
train_dataset = load_dataset(
  "json", 
  data_files="./../data/raw/dataset/python/train.jsonl",
  split="train"
)
train_dataset = train_dataset

We now have to start tokenizing our text. To do so, for now, we're choosing the [Huggin Face tokenizers](https://github.com/huggingface/tokenizers). We're going to use a BPE-Tokenizer, as suggested by a [paper](https://www.researchgate.net/publication/394590942_How_Different_Tokenization_Algorithms_Impact_LLMs_and_Transformer_Models_for_Binary_Code_Analysis) published in 2025. 

ELIMINA DOPO QUELLO SOPRA O SPOSTA IN ALTRO NOTEBOOK

In our dataset we saw that we actually have a column for both code and summarization (docstring) already tokenized. Now, it is probably not going to be the best option, but for now we're gonna build a vocabolary using the already tokenized strings given by our dataset. We're gonna then compare this approach to using e BPE Tokenizer fully trained. 

In [34]:
train_code_tokens = train_dataset['code_tokens']

In [35]:
train_code_tokens[0]

['def',
 'split_phylogeny',
 '(',
 'p',
 ',',
 'level',
 '=',
 '"s"',
 ')',
 ':',
 'level',
 '=',
 'level',
 '+',
 '"__"',
 'result',
 '=',
 'p',
 '.',
 'split',
 '(',
 'level',
 ')',
 'return',
 'result',
 '[',
 '0',
 ']',
 '+',
 'level',
 '+',
 'result',
 '[',
 '1',
 ']',
 '.',
 'split',
 '(',
 '";"',
 ')',
 '[',
 '0',
 ']']

We can now build a vocabolary using gensim's dictionary

In [36]:
from gensim import corpora
code_dictionary = corpora.Dictionary(train_code_tokens)
print(code_dictionary)

Dictionary<1261577 unique tokens: ['";"', '"__"', '"s"', '(', ')']...>


Now that we've created a dictionary let's analyze it a bit.

In [37]:
code_dictionary.num_docs

251820

In [38]:
code_dictionary.num_pos

24692558

In [39]:
code_dictionary.token2id

{'";"': 0,
 '"__"': 1,
 '"s"': 2,
 '(': 3,
 ')': 4,
 '+': 5,
 ',': 6,
 '.': 7,
 '0': 8,
 '1': 9,
 ':': 10,
 '=': 11,
 '[': 12,
 ']': 13,
 'def': 14,
 'level': 15,
 'p': 16,
 'result': 17,
 'return': 18,
 'split': 19,
 'split_phylogeny': 20,
 '"""An error occurred trying to create the output directory\n                           ({}) with message: {}"""': 21,
 '"""One or more directories in the path ({}) do not exist. If\n                           you are specifying a new directory for output, please ensure\n                           all other directories in the path currently exist."""': 22,
 '# ENOENT: No such file or directory': 23,
 '# should not happen with os.makedirs': 24,
 '==': 25,
 'ENOENT': 26,
 'OSError': 27,
 'as': 28,
 'd': 29,
 'else': 30,
 'ensure_dir': 31,
 'errno': 32,
 'except': 33,
 'exists': 34,
 'format': 35,
 'if': 36,
 'makedirs': 37,
 'msg': 38,
 'not': 39,
 'oe': 40,
 'os': 41,
 'path': 42,
 'strerror': 43,
 'try': 44,
 'twdd': 45,
 '"Input file is closed."':

In [40]:
vocab = list(code_dictionary.token2id.keys())
vocab[:10]

['";"', '"__"', '"s"', '(', ')', '+', ',', '.', '0', '1']

In [41]:
code_dictionary.cfs

{14: 266171,
 20: 1,
 3: 2026413,
 16: 17099,
 6: 1546535,
 15: 4317,
 11: 1340132,
 2: 171,
 4: 2026401,
 10: 1163667,
 5: 93429,
 1: 114,
 17: 32917,
 7: 1975616,
 19: 17154,
 18: 278949,
 12: 542795,
 8: 124278,
 13: 542795,
 9: 123283,
 0: 219,
 31: 40,
 29: 15015,
 36: 404201,
 39: 122805,
 41: 43932,
 42: 64954,
 34: 6036,
 44: 44387,
 37: 1336,
 33: 45760,
 27: 2058,
 28: 26573,
 40: 21,
 24: 1,
 23: 1,
 32: 1444,
 25: 88748,
 26: 206,
 38: 17814,
 45: 2,
 22: 1,
 35: 59787,
 30: 105087,
 21: 1,
 43: 208,
 53: 233,
 54: 6,
 57: 6321,
 47: 14,
 55: 2620,
 48: 195357,
 56: 38874,
 52: 6832,
 50: 530,
 59: 68404,
 49: 22702,
 46: 1,
 51: 43002,
 60: 36593,
 58: 12222,
 90: 3,
 95: 115,
 91: 6084,
 82: 724,
 67: 1,
 98: 71846,
 111: 64823,
 63: 331,
 72: 5,
 106: 12804,
 103: 11987,
 112: 64826,
 80: 10,
 97: 18449,
 79: 699,
 89: 149581,
 96: 212262,
 76: 48365,
 61: 471,
 109: 5968,
 73: 1666,
 85: 473,
 87: 1196,
 92: 48608,
 88: 6779,
 81: 31,
 74: 7686,
 66: 1,
 69: 1,
 107: 10

We need to add some special tokens, such as UNK and PAD

In [42]:
special_tokens = {'[UNK]': 0, '[PAD]': 1, '[BOS]': 2, '[EOS]': 3}
code_dictionary.patch_with_special_tokens(special_tokens)
code_dictionary.token2id

{'";"': 1261577,
 '"__"': 1261578,
 '"s"': 1261579,
 '(': 1261580,
 ')': 4,
 '+': 5,
 ',': 6,
 '.': 7,
 '0': 8,
 '1': 9,
 ':': 10,
 '=': 11,
 '[': 12,
 ']': 13,
 'def': 14,
 'level': 15,
 'p': 16,
 'result': 17,
 'return': 18,
 'split': 19,
 'split_phylogeny': 20,
 '"""An error occurred trying to create the output directory\n                           ({}) with message: {}"""': 21,
 '"""One or more directories in the path ({}) do not exist. If\n                           you are specifying a new directory for output, please ensure\n                           all other directories in the path currently exist."""': 22,
 '# ENOENT: No such file or directory': 23,
 '# should not happen with os.makedirs': 24,
 '==': 25,
 'ENOENT': 26,
 'OSError': 27,
 'as': 28,
 'd': 29,
 'else': 30,
 'ensure_dir': 31,
 'errno': 32,
 'except': 33,
 'exists': 34,
 'format': 35,
 'if': 36,
 'makedirs': 37,
 'msg': 38,
 'not': 39,
 'oe': 40,
 'os': 41,
 'path': 42,
 'strerror': 43,
 'try': 44,
 'twdd': 45,
 '"

In [43]:
train_input_ids = [
  [code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]']) for token in sentence]
  for sentence in train_code_tokens
]
print(train_input_ids[0])

[14, 20, 1261580, 16, 6, 15, 11, 1261579, 4, 10, 15, 11, 15, 5, 1261578, 17, 11, 16, 7, 19, 1261580, 15, 4, 18, 17, 12, 8, 13, 5, 15, 5, 17, 12, 9, 13, 7, 19, 1261580, 1261577, 4, 12, 8, 13]


Now we're quickly going to do the same thing for docstring_tokens, the summarization.

In [44]:
train_docstring_tokens = train_dataset['docstring_tokens']
docstring_dictionary = corpora.Dictionary(train_docstring_tokens)
docstring_dictionary.patch_with_special_tokens(special_tokens)
train_labels = [
  [docstring_dictionary.token2id.get(token, docstring_dictionary.token2id['[UNK]']) for token in sentence]
  for sentence in train_docstring_tokens
]
print(train_docstring_tokens[0])
print(train_labels[0])

['Return', 'either', 'the', 'full', 'or', 'truncated', 'version', 'of', 'a', 'QIIME', '-', 'formatted', 'taxonomy', 'string', '.']
[77847, 5, 12, 7, 9, 13, 14, 8, 4, 77846, 77844, 6, 11, 10, 77845]


We're now going to tokenize test and validation sets too, using the same vocabs we built on the train set.

In [45]:
valid_dataset = load_dataset(
  "json",
  data_files="./../data/raw/dataset/python/valid.jsonl",
  split="train"
)
valid_dataset = valid_dataset
valid_code_tokens = valid_dataset['code_tokens']
valid_input_ids = [
  [code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]']) for token in sentence]
  for sentence in valid_code_tokens
]

valid_docstring_tokens = valid_dataset['docstring_tokens']
valid_labels = [
  [docstring_dictionary.token2id.get(token, docstring_dictionary.token2id['[UNK]']) for token in sentence]
  for sentence in valid_docstring_tokens
]

In [46]:
test_dataset = load_dataset(
  "json",
  data_files="./../data/raw/dataset/python/test.jsonl",
  split="train"
)
test_dataset = test_dataset
test_code_tokens = test_dataset['code_tokens']
test_input_ids = [
  [code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]']) for token in sentence]
  for sentence in test_code_tokens
]

test_docstring_tokens = test_dataset['docstring_tokens']
test_labels = [
  [docstring_dictionary.token2id.get(token, docstring_dictionary.token2id['[UNK]']) for token in sentence]
  for sentence in test_docstring_tokens
]

Let's combine the three sets of data and save them on disk

In [47]:
from datasets import Dataset, DatasetDict
tokenized_datasets = DatasetDict({
  'train': Dataset.from_dict({
    'input_ids': train_input_ids,
    'labels': train_labels
  }),
  'valid': Dataset.from_dict({
    'input_ids': valid_input_ids,
    'labels': valid_labels
  }),
  'test': Dataset.from_dict({
    'input_ids': test_input_ids,
    'labels': test_labels
  })
})

tokenized_datasets.save_to_disk('./../data/processed/tokenized_codexglue/')

Saving the dataset (1/1 shards): 100%|██████████| 14918/14918 [00:00<00:00, 2357863.63 examples/s]


And obviously, let's save on disk the dictionary too.

In [48]:
code_dictionary.save('./../data/processed/tokenized_codexglue/code_dictionary.pt')
docstring_dictionary.save('./../data/processed/tokenized_codexglue/docstring_dictionary.pt')